# NDCG: Graded Relevance and Discount Geometry

The narrative companion to `ndcg_discount_geometry.py`, the canonical reference that **owns every
number** this topic reports. We never redefine the math here — we import the module and walk it
section by section. Its prerequisite, `set-metrics-precision-recall-map-mrr`, defined the set-metric
family over **binary** relevance and reframed every metric as an **estimator**; NDCG generalizes that
to **graded** relevance scored under a rank **discount**, and the estimator thread carries straight over.

Run end to end with:

```
uv run --with numpy --with scipy --with jupyter \
  jupyter execute notebooks/ndcg-discount-geometry/01_ndcg_discount_geometry.ipynb
```


In [ ]:
import pathlib, sys

# Path-robust import: find the module regardless of the kernel's working directory. The module itself
# adds every ancestor notebook dir (the multi-vector subtree + the set-metrics prereq) to sys.path.
_here = pathlib.Path.cwd()
for _cand in (_here, _here / "notebooks" / "ndcg-discount-geometry",
              pathlib.Path("notebooks/ndcg-discount-geometry")):
    if (_cand / "ndcg_discount_geometry.py").exists():
        sys.path.insert(0, str(_cand.resolve()))
        break

import numpy as np
import ndcg_discount_geometry as N
from ndcg_discount_geometry import (
    gain_linear, gain_exponential, discount_log2, discount_geometric, discount_reciprocal,
    dcg_at_k, ideal_dcg_at_k, ndcg_at_k, bm25_ndcg_at_k,
    per_query_ndcg, mean_ndcg, metric_summary, LEG_NAMES, TOPK,
)
corpus = N._corpus()
print("corpus:", corpus["n_queries"], "queries x", corpus["n_docs"], "docs; K =", corpus["r_size"])

## 1. From sets to grades

Binary relevance throws away *degree of match*: a perfect 10-K disclosure and a vaguely-related
transcript snippet both count as "relevant." We re-derive **graded** relevance from the same
exact-MaxSim oracle the set-metrics binary qrels came from — the oracle top-K of each query get grades
{1, 2, 3} by **global tertiles** of their oracle score, everything else grade 0. So
`{doc : grade ≥ 1}` is *exactly* the binary relevant set (the shared-baseline anchor), while the
ideal-DCG profile still varies per query.

In [ ]:
# the grade distribution is balanced thirds (global tertiles), and {grade>=1} nests the binary qrels.
allg = [g for q in range(corpus["n_queries"]) for g in corpus["grades"][q].values()]
print("grade distribution:", {gr: int(np.sum(np.array(allg) == gr)) for gr in (1, 2, 3)})
q0 = 0
print("query 0 grades >=1 == binary qrels_set:",
      {d for d, gr in corpus["grades"][q0].items() if gr >= 1} == corpus["qrels_set"][q0])
N.test_grades_nest_binary_qrels()
print("nesting verified for all queries")

## 2. Gain and discount: the two design choices

A hit of grade $g$ at rank $i$ is worth $\text{gain}(g)\cdot\text{discount}(i)$. Both factors are
**conventions**, not derived:

- **Gain** — linear $g$ (Järvelin–Kekäläinen) vs exponential $2^g-1$ (Burges/LETOR, the common modern
  default), which sharply over-weights the top grades ($g{=}3\mapsto 7$ vs $g{=}1\mapsto 1$).
- **Discount** — the celebrated $1/\log_2(i+1)$, vs the geometric $p^{i-1}$ (rank-biased precision),
  vs $1/i$.

In [ ]:
for g in (0, 1, 2, 3):
    print(f"grade {g}: linear gain = {gain_linear(g):.0f}, exponential gain = {gain_exponential(g):.0f}")
print()
for i in (1, 2, 3, 5, 10):
    print(f"rank {i:2d}: log2 = {discount_log2(i):.4f}, geometric(0.85) = {discount_geometric(i):.4f}, "
          f"1/i = {discount_reciprocal(i):.4f}")

## 3. DCG, IDCG, NDCG — and the twin

$\text{DCG@}k = \sum_{i\le k}\text{gain}(g_{\pi(i)})\,\text{discount}(i)$ is the rank-discounted
gain; $\text{IDCG@}k$ is its maximum over orderings (the ideal ranking); and
$\text{NDCG@}k = \text{DCG@}k/\text{IDCG@}k \in [0,1]$.

Under **linear gain + log₂ discount**, our `ndcg_at_k` must equal the imported `bm25.ndcg_at_k`
bit-for-bit — the reused-routine cross-check.

In [ ]:
leg, q = "dense", N.pick_worked_query(corpus)
r, grades = N._ranking(corpus, leg, q), corpus["grades"][q]
print(f"worked query q={q}, ideal grade profile = {corpus['ideal_grades'][q]}")
print(f"{leg} grades in rank order (top {TOPK}): {N.grades_in_rank_order(corpus, leg, q)}")
print(f"DCG@10 (exp gain) = {dcg_at_k(r, grades, TOPK, gain_exponential):.4f}, "
      f"IDCG@10 = {ideal_dcg_at_k(grades, TOPK, gain_exponential):.4f}, "
      f"NDCG@10 = {ndcg_at_k(r, grades, TOPK, gain_exponential):.4f}")
print()
mine = ndcg_at_k(r, grades, TOPK, gain_linear, discount_log2)
theirs = bm25_ndcg_at_k(r, grades, TOPK)
print(f"twin check: ndcg_at_k(linear, log2) = {mine:.6f}  vs  bm25.ndcg_at_k = {theirs:.6f}  "
      f"|diff| = {abs(mine - theirs):.2e}")
N.test_ndcg_matches_bm25_twin()
print("twin verified on every leg/query/cutoff")

## 4. The ideal ranking is optimal: the rearrangement inequality

Write $\text{DCG} = \langle \mathbf{g}_\pi, \mathbf{d}\rangle$, the inner product of the gains *in
ranked order* with the (strictly decreasing) discount vector $\mathbf{d}$. The **rearrangement
inequality** says this inner product is **maximized** by pairing the largest gain with the largest
discount — i.e. sorting gains descending, which is exactly the ideal ranking — and **minimized** by the
ascending (anti-sorted) order. That is *why* IDCG is the ideal and NDCG $\in[0,1]$.

In [ ]:
rng = np.random.default_rng(0)
grades_sorted = sorted(corpus["grades"][q].values(), reverse=True)
gvec = np.array([gain_exponential(g) for g in grades_sorted])
dvec = np.array([discount_log2(i) for i in range(1, len(gvec) + 1)])
ideal = float((np.sort(gvec)[::-1] * dvec).sum())
worst = float((np.sort(gvec) * dvec).sum())
samples = [float((rng.permutation(gvec) * dvec).sum()) for _ in range(2000)]
print(f"ideal (descending gains) = {ideal:.4f}  [the maximum]")
print(f"worst (ascending gains)  = {worst:.4f}  [the minimum]")
print(f"random permutations: max = {max(samples):.4f} <= ideal, min = {min(samples):.4f} >= worst")
N.test_rearrangement_inequality()
print("rearrangement inequality verified across the corpus")

## 5. Discount geometry

The discount is a fixed *measure* on rank positions. Its **shape** is the topic's namesake: how much
weight sits in the head, and how steeply the value of a position falls. A light-tailed geometric
concentrates weight near the top; the heavy-tailed $\log_2$ keeps caring about deep ranks. The
geometric discount also carries a **closed user model** — the reader examines rank $i$ with probability
$p^{i-1}$ and stops with probability $1-p$, so $\mathbb{E}[\text{docs examined}] = 1/(1-p)$ — the
interpretation $\log_2$ lacks.

In [ ]:
for name, disc in (("log2", discount_log2), ("1/i", discount_reciprocal),
                   ("geometric(0.85)", lambda i: discount_geometric(i, 0.85))):
    head = N.discount_weight_in_head(disc, TOPK, corpus["n_docs"])
    print(f"{name:16s} head-mass in top {TOPK} = {head:.4f}  marginal value at rank 1 = "
          f"{N.marginal_value(disc, 1):.4f}")
print(f"\nRBP user model: E[docs examined] at p=0.85 is {N.rbp_expected_docs_examined(0.85):.4f}")
N.test_discount_geometry()
print("discount geometry verified (geometric concentrates more head-mass than log2)")

## 6. NDCG as an estimator

Mean NDCG is a **sample mean** over a finite query set: an estimator with standard error
$\text{std}/\sqrt{n}$. We feed per-query NDCG straight into the prerequisite's `metric_summary`, watch
the SE shrink like $1/\sqrt n$, and ask the motivating question the next topic resolves: *is the gap
real?* The clearest pair of legs separates within our 40 queries; the closest pair's gap is so small it
would take far more.

In [ ]:
for leg in LEG_NAMES:
    s = metric_summary(per_query_ndcg(corpus, leg, TOPK, gain_exponential, discount_log2))
    print(f"{leg:16s} mean NDCG = {s['mean']:.4f}  SE = {s['se']:.4f}  95% CI = "
          f"[{s['ci_lo']:.4f}, {s['ci_hi']:.4f}]")
print("\nSE ~ 1/sqrt(n) (late_interaction): se*sqrt(n) should be ~constant")
for row in N.ndcg_se_scaling(corpus, "late_interaction"):
    print(f"  n={row['n']:3d}  empirical_se={row['empirical_se']:.4f}  se*sqrt(n)={row['se_root_n']:.4f}")
print(f"\nclearest pair separates at n = {N.projected_ndcg_separation_n(corpus, 'lexical', 'late_interaction')}")
print(f"closest pair would need n = {N.projected_ndcg_separation_n(corpus, 'lexical', 'dense', n_max=5000)} "
      f"(overlaps within our {corpus['n_queries']} queries)")
N.test_ndcg_se_scales_as_inv_sqrt_n(); N.test_bootstrap_se_matches_analytic()
N.test_two_leg_separation_contrast()
print("estimator claims verified")

## 7. Convention sensitivity & consistency

Because the gain and discount are conventions, **the verdict can flip with them**. On this corpus the
three legs form a quality ladder (late interaction > dense > lexical) under every convention, so there
is no *aggregate* leg flip — but the convention reverses the verdict on individual queries, and two
constructed examples flip it starkly:

- **Gain flip**: one perfect doc at rank 1 (exp gain wins) vs three marginal docs up top (linear wins).
- **Discount flip**: one hit at rank 1 (steep geometric wins) vs three deeper hits (heavy-tailed log₂ wins).

The honest caveats: NDCG@$k$ truncation can be *inconsistent* (Wang et al. 2013), and these grades are
oracle-score tertiles, a neutral stand-in for human editorial judgments.

In [ ]:
gf, df = N.constructed_gain_flip(), N.constructed_discount_flip()
print(f"gain flip:     exponential -> {gf['exp']['winner']:8s} | linear -> {gf['lin']['winner']:8s} "
      f"(flips = {gf['flips']})")
print(f"discount flip: geometric   -> {df['geo']['winner']:9s}| log2  -> {df['log']['winner']:8s} "
      f"(flips = {df['flips']})")
flip = N.convention_flips_verdict(corpus)
print(f"\non-corpus: aggregate winners identical ({flip['winners']['exp_log']}), but "
      f"{flip['per_query_gain_reversals']} per-query gain reversals")
N.test_convention_flip_constructed(); N.test_convention_flip_corpus_runs()
print("convention-sensitivity claims verified")

## Full verification harness

Every pedagogical claim above is an executable assertion. Running the module's harness re-checks all of
them and prints the constants the interactive lab mirrors.

In [ ]:
N._run_all()